In [1]:
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import json
from pathlib import Path

import pandas as pd
import transformers
from transformers import AutoTokenizer

MAX_LEN = 256
MODELS = {
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased",
}
SPLITS = ["train", "val", "test", "train_pilot"]

ROOT = Path.cwd().parents[1]
SPLIT_DIR = ROOT / "data/split"
PAIR_DIR = ROOT / "data/tokenized"                 # hasil tokenisasi parent + comment
OUT_DIR = ROOT / "data/tokenized_comment"          # hasil tokenisasi comment saja
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
stats = []
for key, name in MODELS.items():
    tok = AutoTokenizer.from_pretrained(name)
    for split in SPLITS:
        df = pd.read_parquet(SPLIT_DIR / f"{split}.parquet")
        comments = df["comment_clean"].tolist()

        full_len = pd.Series([len(ids) for ids in tok(comments)["input_ids"]])   # sebelum dipotong
        enc = tok(comments, truncation=True, max_length=MAX_LEN)

        out = pd.DataFrame({
            "raw_idx": df["raw_idx"].to_numpy(),
            "label": df["label"].to_numpy(),
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
        })
        out["n_tokens"] = out["input_ids"].str.len()

        # Urutan baris harus sama dengan versi parent + comment
        pair = pd.read_parquet(PAIR_DIR / key / f"{split}.parquet", columns=["raw_idx", "label"])
        assert (pair["raw_idx"] == out["raw_idx"]).all() and (pair["label"] == out["label"]).all()

        (OUT_DIR / key).mkdir(parents=True, exist_ok=True)
        out.to_parquet(OUT_DIR / key / f"{split}.parquet", index=False)

        stats.append({
            "Model": key, "Split": split, "Jumlah": len(out),
            "Median token": full_len.median(), "P99 token": full_len.quantile(0.99),
            "Maks token (setelah)": out["n_tokens"].max(),
            "Terpotong": int((full_len > MAX_LEN).sum()),
        })
        print(f"{key:10s} {split:12s} selesai: {len(out):,} comment")

token_stats = pd.DataFrame(stats).set_index(["Model", "Split"]).round(2)
display(token_stats)
token_stats.to_csv(OUT_DIR / "token_stats.csv")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4445 > 512). Running this sequence through the model will result in indexing errors


roberta    train        selesai: 160,000 comment
roberta    val          selesai: 20,000 comment
roberta    test         selesai: 20,000 comment
roberta    train_pilot  selesai: 40,000 comment


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2224 > 512). Running this sequence through the model will result in indexing errors


distilbert train        selesai: 160,000 comment
distilbert val          selesai: 20,000 comment
distilbert test         selesai: 20,000 comment
distilbert train_pilot  selesai: 40,000 comment


Jumlah  Median token  P99 token  Maks token (setelah)  \
Model      Split                                                                
roberta    train        160000          12.0       44.0                   256   
           val           20000          12.0       44.0                   256   
           test          20000          12.0       44.0                   256   
           train_pilot   40000          12.0       44.0                   256   
distilbert train        160000          12.0       44.0                   256   
           val           20000          12.0       44.0                   256   
           test          20000          12.0       44.0                   256   
           train_pilot   40000          12.0       44.0                   256   

                        Terpotong  
Model      Split                   
roberta    train               12  
           val                  3  
           test                 3  
           train_pilot          3  
distilbert train               10  
           val                  2  
           test                 2  
           train_pilot          3

In [3]:
for key, name in MODELS.items():
    tok = AutoTokenizer.from_pretrained(name)
    t = pd.read_parquet(OUT_DIR / key / "train.parquet")
    print(f"{key}: {t.shape} |", tok.decode(t["input_ids"].iloc[0]))

config = {
    "input": "comment_clean saja (tanpa parent)",
    "max_length": MAX_LEN,
    "models": MODELS,
    "transformers_version": transformers.__version__,
}
(OUT_DIR / "tokenization_config.json").write_text(json.dumps(config, indent=2))


roberta: (160000, 5) | <s>weird i live in nc and have never seen one</s>
distilbert: (160000, 5) | [CLS] weird i live in nc and have never seen one [SEP]


201